<a href="https://www.kaggle.com/code/samratrm/adaboost-scratch-code?scriptVersionId=339729754" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# AdaBoost Scratch Code 

#### **Objective:**

To build and understand a custom AdaBoost classifier from scratch using weighted Gini Impurity with NumPy and Pandas and evaluate model performance and intuitively visualize key classification metrics.

### How it Works: 

AdaBoost means Adaptive Boosting which is a ensemble learning technique that combines multiple weak classifiers to create a strong classifier. It works by sequentially adding classifiers to correct the errors made by previous models giving more weight to the misclassified data points.

[StatQuest_video_link](https://www.youtube.com/watch?v=LsK-xG1cLYA)

[GeekForGeeks_AdaBoost_Scratch_code_blog](https://www.geeksforgeeks.org/machine-learning/implementing-the-adaboost-algorithm-from-scratch/)

### **Advantages**
- Turns weak learners (e.g., depth-1 stumps) into a strong classifier — high accuracy with a very simple base model
- Few hyperparameters to tune (number of rounds T, learning rate)
- No feature scaling needed; handles mixed feature types
- Does implicit feature selection when using stumps
- Tends to keep improving test error even after training error hits zero (margin maximization)
- Tiny model size, very fast prediction

### **Disadvantages**
- Highly sensitive to noisy data and outliers — exponential loss keeps boosting the weight of misclassified points
- Sequential by nature, so rounds can't be parallelized (unlike Random Forest)
- Can overfit if T is too large, especially with label noise
- Requires each weak learner to be better than random (error < 0.5); it breaks down otherwise
- Vanilla AdaBoost is binary only — multiclass needs SAMME / SAMME.R
- Less interpretable than a single tree

### **Time complexity** (n = samples, d = features, T = rounds)

| Phase | Complexity |
|---|---|
| Training (stumps, naive) | O(T · d · n log n) |
| Training (pre-sorted features) | O(T · d · n) |
| Weight update per round | O(n) — negligible |
| Inference per sample | O(T) with stumps, O(T · h) for depth-h trees |
| Space | O(T) — stores each stump + its weight α |

The one-line intuition: training cost = T × (cost of training one weak learner), inference cost = T × (cost of one weak learner's prediction).

## Scratch Code

In [19]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

In [10]:
class AdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = [] # in SQ vid: the "amount of say"
        self.models = [] # all the stumps 
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        w = np.ones(n_samples) / n_samples

        for _ in range(self.n_estimators):
            model = DecisionTreeClassifier(max_depth=1) # Stump 
            model.fit(X,y, sample_weight=w)  
            preds = model.predict(X)
            total_err = np.sum(w * (preds != y)) / np.sum(w) # (0,1)
            alpha = 0.5 * np.log((1-total_err)/(total_err + 1e-10)) # Amount of say (inf, -inf)

            self.models.append(model)
            self.alphas.append(alpha)

            # update weights 
            w *= np.exp(-alpha * y * preds)
            w/= np.sum(w)

    def predict(self, X):
        strong_preds = np.zeros(X.shape[0])

        for model, alpha in zip(self.models, self.alphas):
            preds = model.predict(X)
            strong_preds += alpha * preds
        
        return np.sign(strong_preds).astype(int)
        

### The four formulas, in plain words

**1. Starting weights**

```
weight of each sample = 1 / (number of samples)
```

Everyone starts equally important.

$$w_i^{(1)} = \frac{1}{n}, \quad i = 1, \dots, n$$



**2. Weighted error of the stump**

```
                sum of weights of the samples it got WRONG
total error  =  ------------------------------------------
                        sum of ALL weights
```

Since the weights always add up to 1, this simplifies to:

```
total error = sum of weights of the misclassified samples
```

If a stump misses samples carrying 30% of the total weight, `total_error = 0.3`. Note it's not "how many did it miss" but **"how much weight did it miss"** — missing one heavy sample can hurt more than missing three light ones.

$$\varepsilon_t = \frac{\sum_{i=1}^{n} w_i \cdot \mathbb{1}[h_t(x_i) \neq y_i]}{\sum_{i=1}^{n} w_i}$$


**3. Amount of say (alpha)**

```
amount of say = 0.5 × ln( (1 - total error) / total error )
```

Read the fraction as **(weight it got right) ÷ (weight it got wrong)**.

| total error | amount of say | meaning |
|---|---|---|
| 0.01 | +2.3 | almost always right → loud vote |
| 0.5 | 0 | coin flip → no vote at all |
| 0.9 | −1.1 | usually wrong → vote gets flipped |

αt=12ln(1−εtεt)\alpha_t = \frac{1}{2} \ln\!\left(\frac{1 - \varepsilon_t}{\varepsilon_t}\right)



**4. Weight update**

```
new weight = old weight × e^(± amount of say)
```

with the sign chosen by whether the stump was right:

```
got it right  →  new weight = old weight × e^(−amount of say)   (shrinks)
got it wrong  →  new weight = old weight × e^(+amount of say)   (grows)
```

Then rescale so they add to 1 again:

```
new weight = new weight / (sum of all new weights)
```

The code writes both cases as one line using `y * preds` (which is `+1` when correct, `−1` when wrong) purely to avoid an if/else.

w(t+1)i=w(t)i⋅e−αtyiht(xi)Zt,Zt=n∑j=1w(t)je−αtyjht(xj)w_i^{(t+1)} = \frac{w_i^{(t)} \cdot e^{-\alpha_t \, y_i \, h_t(x_i)}}{Z_t}, \qquad Z_t = \sum_{j=1}^{n} w_j^{(t)} e^{-\alpha_t y_j h_t(x_j)}

#### The weight update in simple terms

Labels are $-1/+1$, so $y_i \cdot h_t(x_i)$ is:

- **$+1$** if the stump got sample $i$ right
- **$-1$** if it got it wrong

Plug that into the exponent:

| Case | Multiplier | Effect |
|---|---|---|
| Correct | $e^{-\alpha}$ | < 1 → weight **shrinks** |
| Wrong | $e^{+\alpha}$ | > 1 → weight **grows** |

So: **misclassified samples get heavier, correctly classified ones get lighter.** The next stump is trained on this reweighted data, so it's forced to focus on what the previous one messed up. Dividing by $Z_t$ just rescales everything back to sum to 1 so it stays a proper distribution.

$\alpha_t$ controls how *aggressive* the reweighting is — a stump with low error gets a large $\alpha$, so it shuffles the weights a lot; a stump near 50% error gets $\alpha \approx 0$, so weights barely move (its opinion isn't trusted).



**5. Final prediction**

```
score = (say₁ × vote₁) + (say₂ × vote₂) + ... + (sayT × voteT)

prediction = +1 if score > 0, else −1
```

A weighted popular vote where each stump's ballot counts as much as its "amount of say."

H(x)=sign(T∑t=1αtht(x))H(x) = \operatorname{sign}\!\left(\sum_{t=1}^{T} \alpha_t \, h_t(x)\right)

In [15]:
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

adaBoost = AdaBoost()
adaBoost.fit(X_train, y_train)

preds = adaBoost.predict(X_test)

accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds)
recall = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
try:
    roc_auc = roc_auc_score(y_test, preds)
except ValueError:
    roc_auc = 'Undefined (requires probability scores)'

print(f"Accuracy: {accuracy * 100}%")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Accuracy: 86.0%
Precision: 0.8691588785046729
Recall: 0.8691588785046729
F1 Score: 0.8691588785046729
ROC-AUC: 0.8593106220480354


In [22]:
import plotly.express as px

pentagon_metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
pentagon_values = [accuracy, precision, recall, f1, roc_auc] 

df_pentagon = pd.DataFrame({
    'Metric': pentagon_metrics,
    'Value': pentagon_values
})

fig_pentagon = px.line_polar(
    df_pentagon, 
    r='Value', 
    theta='Metric', 
    line_close=True, 
    range_r=[0, 1],
    title="Ada Boost Performance (Pentagon)"
)
fig_pentagon.update_traces(fill='toself')
fig_pentagon.show()